In [ ]:
import pandas as pd
import os

# CSV 파일들이 위치한 폴더 경로
folder_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵'

# 폴더 내의 모든 CSV 파일 리스트를 가져오기
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# 빈 데이터프레임 생성
all_data = pd.DataFrame()

# 모든 CSV 파일 읽어서 데이터프레임에 추가
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    data = pd.read_csv(file_path)
    all_data = pd.concat([all_data, data], ignore_index=True)

# 열 중복 제거 (기본적으로 모든 중복된 열을 제거하는 것이 아니라, 열 이름이 같을 때 첫 번째 열만 유지합니다)
all_data = all_data.loc[:,~all_data.columns.duplicated()]

# 중복된 행 제거 (선택적으로 제거하려는 경우)
# all_data.drop_duplicates(inplace=True)

# 최종 결과를 새로운 CSV 파일로 저장
output_file = os.path.join(folder_path, 'merged_data.csv')
all_data.to_csv(output_file, index=False, encoding='utf-8-sig')  # UTF-8 인코딩으로 저장

print(f'CSV 파일이 성공적으로 병합되었습니다. 결과 파일: {output_file}')


In [ ]:
import pandas as pd
import os

# CSV 파일들이 위치한 폴더 경로
folder_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵'

# 폴더 내의 모든 CSV 파일 리스트를 가져오기
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# 빈 데이터프레임 생성
all_data = pd.DataFrame()

# 모든 CSV 파일 읽어서 데이터프레임에 추가
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    data = pd.read_csv(file_path)
    all_data = pd.concat([all_data, data], ignore_index=True)

# 열 중복 제거 (기본적으로 모든 중복된 열을 제거하는 것이 아니라, 열 이름이 같을 때 첫 번째 열만 유지합니다)
all_data = all_data.loc[:,~all_data.columns.duplicated()]

# 행 중복 제거 (모든 중복된 행을 제거합니다)
all_data.drop_duplicates(inplace=True)

# 최종 결과를 새로운 CSV 파일로 저장
output_file = os.path.join(folder_path, 'merged_data_cleaned.csv')
all_data.to_csv(output_file, index=False, encoding='utf-8-sig')  # UTF-8 인코딩으로 저장

print(f'CSV 파일이 성공적으로 병합되고 중복이 제거되었습니다. 결과 파일: {output_file}')


In [ ]:
pip install geopy

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
import time  # API 요청 간의 지연 시간을 추가하는 데 사용

# CSV 파일의 경로 설정
input_file = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\merged_data_cleaned.csv'
output_file = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\merged_data_with_geocoding.csv'

# CSV 파일 읽기
data = pd.read_csv(input_file)

# Nominatim 지오코더 객체 생성
geolocator = Nominatim(user_agent="geoapiExercises")

# 위도와 경도를 저장할 빈 열 생성
data['Latitude'] = None
data['Longitude'] = None

# 주소를 기반으로 위도와 경도 정보 추가
for idx, row in data.iterrows():
    address = row['도로명 주소']  # '주소열이름'은 실제 주소 열의 이름으로 변경해야 합니다.
    if pd.notna(address):
        try:
            location = geolocator.geocode(address, timeout=10)
            if location:
                data.at[idx, 'Latitude'] = location.latitude
                data.at[idx, 'Longitude'] = location.longitude
            else:
                data.at[idx, 'Latitude'] = None
                data.at[idx, 'Longitude'] = None
        except Exception as e:
            print(f'Error geocoding address {address}: {e}')
            data.at[idx, 'Latitude'] = None
            data.at[idx, 'Longitude'] = None
        time.sleep(1)  # Nominatim API 요청에 대한 Rate Limit을 준수하기 위한 지연 시간 추가

# 최종 결과를 새로운 CSV 파일로 저장
data.to_csv(output_file, index=False, encoding='utf-8-sig')  # UTF-8 인코딩으로 저장

print(f'위도와 경도 정보가 성공적으로 추가되었습니다. 결과 파일: {output_file}')


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\merged_data_cleaned.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_geocoded전체.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder")

# 지오코딩 함수 정의
def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        return geocode_address(address)

# 위도와 경도 컬럼 추가
df['위도'], df['경도'] = zip(*df['도로명 주소'].apply(geocode_address))

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\merged_data_cleaned.csv'
output_file_path = r'C:\Users\kfq\Downloads\Final_Project\카카오맵 크롤링\쏘카 지점 카카오맵\socar_locations_geocoded전체3.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder")

# 지오코딩 함수 정의
def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        return geocode_address(address)

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    # 쏘카존 지점으로 먼저 지오코딩 시도
    lat, lon = geocode_address(row['장소'])
    if lat is None or lon is None:
        # 실패하면 도로명 주소로 지오코딩 시도
        lat, lon = geocode_address(row['도로명 주소'])
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)
